In [ ]:
!pip install transformers accelerate mlflow torch scikit-learn evaluate 


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM
import mlflow
from transformers.integration import MLflowCallback
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import load_dataset, DatasetDict 
from sklearn.model_selection import train_test_split
import pandas as pd
tokenizer = AutoTokenizer.from_pretrained("Shushant/nepaliBERT")
model = AutoModelForMaskedLM.from_pretrained("Shushant/nepaliBERT")

/Users/urgensingtan/Desktop/preprocessing/enviroment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an e

In [ ]:
df = pd.read_parquet("nepali_sentiment_dataset.parquet")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
raw_datasets = load_dataset("parquet", data_files={"train": train_df, "validation": val_df})
def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)
tokenized = raw_datasets.map(tokenize_fn, batched=True)
tokenized = tokenized.remove_columns([c for c in tokenized["train"].column_names if c not in ("input_ids","attention_mask","label")])
tokenized.set_format(type="torch")
train_dataset = tokenized["train"]
eval_dataset = tokenized["validation"]

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

In [ ]:
# ...existing code...
training_args = TrainingArguments(
    output_dir="./outputs",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    fp16=True,
    dataloader_num_workers=8,
    dataloader_pin_memory=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to=["mlflow"],   # enable reporting to MLflow
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[MLflowCallback()]
)

# Optionally create an explicit MLflow run to attach extra tags/params
with mlflow.start_run() as run:
    mlflow.log_param("model_name", "Shushant/nepaliBERT")
    mlflow.log_param("num_labels", 2)
    trainer.train()
    metrics = trainer.evaluate()
    mlflow.log_metrics(metrics)
    trainer.save_model("./outputs/best_model")
# ...existing code...